In [1]:
!pip install -q -U transformers peft accelerate trl bitsandbytes datasets pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 85.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-pr

In [2]:
import os
import shutil
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    AutoConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
 
torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False)
 
print("=" * 60)
print("STEP 1 — LOADING DATA")
print("=" * 60)
 
file_path = "/kaggle/input/datasets/mrunmayeepotdar/codex-dataset/codex_clean.csv"
df = pd.read_csv(file_path).sample(frac=1, random_state=42).reset_index(drop=True)
 
def format_prompt(row):
    clean_code = str(row["code"]).replace("```python", "").replace("```", "").strip()
    return (
        "### Instruction: Write Python code to solve the math problem. "
        "Store the answer in 'result'.\n"
        f"### Question:\n{row['question']}\n"
        f"### Code:\n```python\n{clean_code}\n```"
    )
 
df["text"] = df.apply(format_prompt, axis=1)
dataset = Dataset.from_pandas(df[["text"]])          
print(f"  Dataset size: {len(dataset)} rows")

STEP 1 — LOADING DATA
  Dataset size: 8000 rows


In [3]:
print("\nSTEP 2 — TOKENIZER")
model_id = "microsoft/Phi-3-mini-4k-instruct"
 
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"


STEP 2 — TOKENIZER


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [4]:
print("\nSTEP 3 — CONFIG")
config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
 
rope = getattr(config, "rope_scaling", None) or {}
if not isinstance(rope, dict):
    rope = {}
rope.setdefault("type", "longrope")
if rope["type"] == "longrope":
    rope.setdefault("short_factor", 1.0)
    rope.setdefault("long_factor",  1.0)
config.rope_scaling = rope


STEP 3 — CONFIG


In [5]:
print("\nSTEP 4 — MODEL LOAD")
 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   
)
 
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",            
    torch_dtype=torch.float16,
)

for _, p in model.named_parameters():
    if p.dtype == torch.bfloat16:
        p.data = p.data.to(torch.float16)
 
model = prepare_model_for_kbit_training(model)
 
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
 
for _, p in model.named_parameters():
    if p.dtype == torch.bfloat16:
        p.data = p.data.to(torch.float16)
 
model.print_trainable_parameters()


STEP 4 — MODEL LOAD


modeling_phi3.py: 0.00B [00:00, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

trainable params: 25,165,824 || all params: 3,846,245,376 || trainable%: 0.6543


In [6]:
print("\nSTEP 5 — TRAINING")
sft_config = SFTConfig(
    output_dir="/kaggle/working/phi3-math-agent",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=2,
    fp16=False,          
    bf16=False,          
    max_grad_norm=1.0,   
    optim="paged_adamw_8bit",
    report_to="none",
    dataset_text_field="text",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
)
trainer.train()


STEP 5 — TRAINING


/tmp/ipykernel_23/2306419115.py:2: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  sft_config = SFTConfig(


Adding EOS to train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is de

Step,Training Loss
10,9.628268
20,6.817521
30,5.356248
40,4.845966
50,4.306499
60,4.162704
70,4.013916
80,3.897983
90,3.892266
100,3.445546


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=2000, training_loss=2.2589624547958373, metrics={'train_runtime': 19270.5304, 'train_samples_per_second': 0.83, 'train_steps_per_second': 0.104, 'total_flos': 1.0045204726796698e+17, 'train_loss': 2.2589624547958373, 'epoch': 2.0})

In [7]:
print("\nSTEP 6 — SAVING & ZIPPING")
 
save_dir = "/kaggle/working/final-neuro-symbolic-adapter"
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
shutil.make_archive(save_dir, "zip", save_dir)
 
print("\nTRAINING COMPLETE — adapter saved and zipped!")


STEP 6 — SAVING & ZIPPING

TRAINING COMPLETE — adapter saved and zipped!
